## NCAA Bracket Model — Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
def show_columns(df):
    for col in df.columns:
        print(col)

### Step 1: Build Team Season Averages

In [ ]:
season_stats = pd.read_csv("../data/raw/MRegularSeasonDetailedResults.csv")
season_stats.head()

In [ ]:
show_columns(season_stats)

In [ ]:
winners = season_stats[['Season', 'WTeamID', 'WScore', 'LScore', 'WFGM', 'WFGA', 'WFGM3', 'WFGA3', 'WFTM', 'WFTA', 'WAst', 'WTO', 'WStl', 'WBlk', 'WPF']]
winners.rename(columns={'WTeamID': 'TeamID', 'WScore': 'Score', 'LScore': 'OppScore', 'WFGM': 'FGM', 'WFGA': 'FGA', 'WFGM3': 'FGM3', 'WFGA3': 'FGA3', 'WFTM': 'FTM', 'WFTA': 'FTA', 'WAst': 'Ast', 'WTO': 'TO', 'WStl': 'Stl', 'WBlk': 'Blk', 'WPF': 'PF'}, inplace=True)
winners['Win'] = 1
winners.head()

In [ ]:
losers = season_stats[['Season', 'LTeamID', 'LScore', 'WScore', 'LFGM', 'LFGA', 'LFGM3', 'LFGA3', 'LFTM', 'LFTA', 'LAst', 'LTO', 'LStl', 'LBlk', 'LPF']]
losers.rename(columns={'LTeamID': 'TeamID', 'LScore': 'Score', 'WScore': 'OppScore', 'LFGM': 'FGM', 'LFGA': 'FGA', 'LFGM3': 'FGM3', 'LFGA3': 'FGA3', 'LFTM': 'FTM', 'LFTA': 'FTA', 'LAst': 'Ast', 'LTO': 'TO', 'LStl': 'Stl', 'LBlk': 'Blk', 'LPF': 'PF'}, inplace=True)
losers['Win'] = 0
losers.head()

In [ ]:
team_stats = pd.concat([winners, losers], ignore_index=True)
team_stats.head()

In [ ]:
# Verifying the proper amount of rows
print(f"Number of rows in winners: {len(winners)}\nNumber of rows in losers: {len(losers)}\nTotal rows in team_stats: {len(team_stats)}")
if len(winners) + len(losers) == len(team_stats) and len(season_stats) * 2 == len(team_stats):
    print("The number of rows in team_stats is correct.")


In [ ]:
season_averages = team_stats.groupby(['Season', 'TeamID']).mean().reset_index().sort_values(by=['Season', 'TeamID'])    
season_averages.head()

In [ ]:
season_averages.shape

In [ ]:
checkwin = season_averages[(season_averages['Win'] < 0) | (season_averages['Win'] > 1)]
if len(checkwin) > 0:
    print("There are invalid values in the 'Win' column.")
else:
    print("All values in the 'Win' column are valid (0 or 1).")

### Step 2: Clean Seed Data

In [ ]:
seeds = pd.read_csv("../data/raw/MNCAATourneySeeds.csv")
seeds.head()

In [ ]:
seed_clean = seeds.copy()
seed_clean['Seed'] = seed_clean['Seed'].str.replace(r'[a-zA-Z]', '', regex=True).astype(int)
seed_clean = seed_clean[['Season', 'TeamID', 'Seed']]
seed_clean.head()

In [ ]:
if (seed_clean['Seed'] <= 16).all() and (seed_clean['Seed'] >= 1).all():
    print("All seed values are valid (between 1 and 16).")

### Step 3: Build Rankings Table

In [ ]:
massey = pd.read_csv("../data/raw/MMasseyOrdinals.csv")
massey.head()

In [ ]:
massey_clean = massey.copy()
massey_clean = massey_clean[(massey_clean['RankingDayNum'] == 133) & massey_clean['SystemName'].isin(['POM', 'SAG', 'MOR', 'DUN'])].reset_index(drop=True)
massey_clean = massey_clean[['Season', 'TeamID', 'SystemName', 'OrdinalRank']]      
massey_clean.head()


In [ ]:
massey_pivot = massey_clean.pivot(index = ['Season', 'TeamID'], columns = 'SystemName', values = 'OrdinalRank').reset_index()
massey_pivot.columns.name = None
massey_pivot.head()

In [ ]:
massey_pivot.isnull().sum()

In [ ]:
# Cleaning up massey data
massey_pivot['DUN'] = massey_pivot.groupby('Season')['DUN'].transform(lambda x: x.fillna(x.median())) 
massey_pivot['MOR'] = massey_pivot.groupby('Season')['MOR'].transform(lambda x: x.fillna(x.median()))
massey_pivot['POM'] = massey_pivot.groupby('Season')['POM'].transform(lambda x: x.fillna(x.median()))
massey_pivot['SAG'] = massey_pivot.groupby('Season')['SAG'].transform(lambda x: x.fillna(x.median()))
massey_pivot.head() 

In [ ]:
massey_pivot.isnull().sum()

In [ ]:
massey_pivot.groupby('Season')['DUN'].count()

In [ ]:
massey_pivot.groupby('Season')['SAG'].count()

In [ ]:
#Handeling Missing Values in DUN
dun_median = massey_pivot.groupby('Season')['DUN'].median().sort_index().ffill()
print(dun_median)
season_medians = massey_pivot['Season'].map(dun_median)
massey_pivot['DUN'] = massey_pivot['DUN'].fillna(season_medians)
massey_pivot.isnull().sum()

In [ ]:
#Handeling Missing Values in SAG
sag_median = massey_pivot.groupby('Season')['SAG'].median().sort_index().ffill()
print(sag_median)
season_medians = massey_pivot['Season'].map(sag_median)
massey_pivot['SAG'] = massey_pivot['SAG'].fillna(season_medians)
massey_pivot.isnull().sum()